# Analyze structured HK epsilon robustness

Pairs bounded and unbounded alpha runs at each confidence radius. This is a robustness diagnostic, not a parameter search that automatically selects whichever environment gives the largest score.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)

HERE = Path.cwd()
RESULTS = HERE / "results"
BOUNDED_DIR = RESULTS / "hk_eps_b"
UNBOUNDED_DIR = RESULTS / "hk_eps_u"
OUT = RESULTS / "hk_epsilon_robustness_comparison"
OUT.mkdir(parents=True, exist_ok=True)

for label, p in [("bounded", BOUNDED_DIR), ("unbounded", UNBOUNDED_DIR)]:
    f = p / "combined_summary.csv"
    if not f.exists():
        raise FileNotFoundError(f"Missing {label} merged results: {f}")

b = pd.read_csv(BOUNDED_DIR / "combined_summary.csv")
u = pd.read_csv(UNBOUNDED_DIR / "combined_summary.csv")

required = {
    "environment_id", "family", "hk_epsilon", "policy", "mean_end",
    "initial_active_edge_fraction", "initial_bridge_opinion_gap", "initial_bridge_active",
}
for label, df in [("bounded", b), ("unbounded", u)]:
    missing = sorted(required - set(df.columns))
    if missing:
        raise RuntimeError(f"{label} results missing columns: {missing}")


def ci95(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    m = float(x.mean())
    if n == 1:
        return m, m, m
    se = float(x.std(ddof=1) / np.sqrt(n))
    q = float(student_t.ppf(0.975, n - 1))
    return m, m - q * se, m + q * se


def env_table(df, prefix):
    learned = (
        df[df["policy"].astype(str).str.contains("learned", case=False, na=False)]
        .groupby(
            [
                "environment_id", "family", "hk_epsilon",
                "initial_active_edge_fraction", "initial_bridge_opinion_gap",
                "initial_bridge_active",
            ],
            as_index=False,
        )
        .agg(
            learned_mean=("mean_end", "mean"),
            learned_seed_std=("mean_end", "std"),
            prediction_ratio=("model_over_identity", "mean"),
        )
    )

    def baseline(policy, name):
        return (
            df[df["policy"].astype(str).str.fullmatch(policy, case=False, na=False)]
            .groupby("environment_id", as_index=False)["mean_end"]
            .mean()
            .rename(columns={"mean_end": name})
        )

    out = (
        learned
        .merge(baseline("uniform", "uniform"), on="environment_id", validate="one_to_one")
        .merge(baseline("true_graph", "true_graph"), on="environment_id", validate="one_to_one")
        .merge(baseline("no_control", "no_control"), on="environment_id", validate="one_to_one")
    )
    return out.rename(columns={
        "learned_mean": f"{prefix}_learned",
        "learned_seed_std": f"{prefix}_seed_std",
        "prediction_ratio": f"{prefix}_prediction_ratio",
        "uniform": f"{prefix}_uniform",
        "true_graph": f"{prefix}_true_graph",
        "no_control": f"{prefix}_no_control",
    })


be = env_table(b, "bounded")
ue = env_table(u, "unbounded")

keys = [
    "environment_id", "family", "hk_epsilon",
    "initial_active_edge_fraction", "initial_bridge_opinion_gap",
    "initial_bridge_active",
]
pair = be.merge(ue, on=keys, validate="one_to_one")

for name in ["uniform", "true_graph", "no_control"]:
    d = np.abs(pair[f"bounded_{name}"] - pair[f"unbounded_{name}"])
    print(f"baseline max |bounded-unbounded| for {name}: {d.max():.3e}")

pair["delta_unbounded_minus_bounded"] = pair["unbounded_learned"] - pair["bounded_learned"]
pair["bounded_gain_vs_uniform"] = pair["bounded_learned"] - pair["bounded_uniform"]
pair["unbounded_gain_vs_uniform"] = pair["unbounded_learned"] - pair["unbounded_uniform"]
pair["unbounded_gap_to_true"] = pair["unbounded_true_graph"] - pair["unbounded_learned"]
pair["delta_prediction_ratio"] = (
    pair["unbounded_prediction_ratio"] - pair["bounded_prediction_ratio"]
)
pair.to_csv(OUT / "environment_level_comparison.csv", index=False)

rows = []
for eps, g in pair.groupby("hk_epsilon", sort=True):
    dm, dlo, dhi = ci95(g["delta_unbounded_minus_bounded"])
    um, ulo, uhi = ci95(g["unbounded_learned"])
    gm, glo, ghi = ci95(g["unbounded_gain_vs_uniform"])
    rows.append({
        "hk_epsilon": float(eps),
        "n_environments": len(g),
        "initial_bridge_active_rate": float(g["initial_bridge_active"].mean()),
        "initial_active_edge_fraction_mean": float(g["initial_active_edge_fraction"].mean()),
        "bounded_learned_mean": float(g["bounded_learned"].mean()),
        "unbounded_learned_mean": um,
        "unbounded_learned_ci_low": ulo,
        "unbounded_learned_ci_high": uhi,
        "delta_unbounded_minus_bounded": dm,
        "delta_ci_low": dlo,
        "delta_ci_high": dhi,
        "unbounded_gain_vs_uniform_mean": gm,
        "unbounded_gain_ci_low": glo,
        "unbounded_gain_ci_high": ghi,
        "unbounded_prediction_ratio_mean": float(g["unbounded_prediction_ratio"].mean()),
        "bounded_prediction_ratio_mean": float(g["bounded_prediction_ratio"].mean()),
        "environment_win_rate_unbounded_vs_bounded": float(
            (g["delta_unbounded_minus_bounded"] > 0).mean()
        ),
    })
overall = pd.DataFrame(rows)
overall.to_csv(OUT / "summary_by_epsilon.csv", index=False)

fam_rows = []
for (eps, family), g in pair.groupby(["hk_epsilon", "family"], sort=True):
    dm, dlo, dhi = ci95(g["delta_unbounded_minus_bounded"])
    fam_rows.append({
        "hk_epsilon": float(eps),
        "family": family,
        "n_environments": len(g),
        "initial_bridge_active_rate": float(g["initial_bridge_active"].mean()),
        "bounded_learned_mean": float(g["bounded_learned"].mean()),
        "unbounded_learned_mean": float(g["unbounded_learned"].mean()),
        "delta_unbounded_minus_bounded": dm,
        "delta_ci_low": dlo,
        "delta_ci_high": dhi,
        "unbounded_gain_vs_uniform_mean": float(g["unbounded_gain_vs_uniform"].mean()),
        "unbounded_prediction_ratio_mean": float(g["unbounded_prediction_ratio"].mean()),
    })
by_family = pd.DataFrame(fam_rows)
by_family.to_csv(OUT / "summary_by_epsilon_and_family.csv", index=False)

print("\n=== OVERALL BY EPSILON ===")
display(overall.round(6))

print("\n=== BY EPSILON AND FAMILY ===")
display(by_family.round(6))

# Useful, non-cherry-picking selection view:
# prioritize epsilon regions where (1) the environment is not completely
# disconnected by confidence, (2) unbounded does not show a large systematic
# regression, and (3) the learned policy still has nontrivial room vs uniform.
selection = overall.copy()
selection["abs_alpha_gap"] = selection["delta_unbounded_minus_bounded"].abs()
selection = selection.sort_values(
    ["abs_alpha_gap", "unbounded_gain_vs_uniform_mean"],
    ascending=[True, False],
)
selection.to_csv(OUT / "selection_diagnostic_sorted.csv", index=False)
print("\n=== SELECTION DIAGNOSTIC (small alpha gap first; not an automatic recommendation) ===")
display(selection.round(6))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(overall["hk_epsilon"], overall["delta_unbounded_minus_bounded"], marker="o")
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("HK epsilon")
ax.set_ylabel("Unbounded - bounded final mean")
ax.set_title("Structured HK: alpha simplification gap vs confidence radius")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(overall["hk_epsilon"], overall["bounded_learned_mean"], marker="o", label="bounded alpha")
ax.plot(overall["hk_epsilon"], overall["unbounded_learned_mean"], marker="o", label="unbounded alpha")
ax.set_xlabel("HK epsilon")
ax.set_ylabel("Final mean opinion")
ax.set_title("Structured HK performance across confidence radius")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for family, g in by_family.groupby("family"):
    g = g.sort_values("hk_epsilon")
    ax.plot(
        g["hk_epsilon"],
        g["delta_unbounded_minus_bounded"],
        marker="o",
        label=family,
    )
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("HK epsilon")
ax.set_ylabel("Unbounded - bounded final mean")
ax.set_title("Structured HK simplification gap by topology family")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    overall["hk_epsilon"],
    overall["initial_bridge_active_rate"],
    marker="o",
    label="bridge initially active",
)
ax.plot(
    overall["hk_epsilon"],
    overall["initial_active_edge_fraction_mean"],
    marker="o",
    label="active graph-edge fraction",
)
ax.set_xlabel("HK epsilon")
ax.set_ylabel("Fraction")
ax.set_ylim(-0.02, 1.02)
ax.set_title("How the HK environment changes with epsilon")
ax.legend()
plt.tight_layout()
plt.show()

print("\nOutputs:", OUT)
print("Main tables:")
print(" ", OUT / "summary_by_epsilon.csv")
print(" ", OUT / "summary_by_epsilon_and_family.csv")
print(" ", OUT / "selection_diagnostic_sorted.csv")
